# PathSnap — XGBoost candidate road-segment classifier
**Google Colab | GPS map matching | reproducible training and Bayesian tuning**

Upload both `cleaned_train(1).zip` and `Student folder.zip`. The cleaned ZIP supplies the 80 training GPS tracks; the original ZIP supplies nodes, directed arcs, reference routes and the 10 student-test trips. Arc IDs are local to each trip, so we learn a transferable binary candidate score, not a global arc-ID classifier.

The `.route` file contains a sequence of arc IDs rather than observation-level labels. We derive approximate observation labels by projecting GPS points onto the reference route, with a small heading-aware tie-break. These are **proxy labels**, not independent pointwise ground truth. Reference-route data is never an inference feature. The evaluator partition is not read. This notebook is the XGBoost experiment, not the full five-model or HMM/API deliverable. Run the experiments yourself, inspect the outputs, and disclose AI assistance as required.

In [ ]:
%pip -q install xgboost optuna

In [ ]:
import io, zipfile, time, json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, log_loss, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier
import optuna

SEED = 42
K = 12
TRAIN_STRIDE = 2  # use 1 for all cleaned observations, at greater computational cost
N_TRIALS = 8      # increase later if time permits
N_JOBS = min(4, os.cpu_count() or 2)
OUT = Path('pathsnap_xgb_results'); OUT.mkdir(exist_ok=True)

In [ ]:
from google.colab import files
from pathlib import Path

if not list(Path('.').glob('cleaned_train*.zip')) or not Path('Student folder.zip').exists():
    files.upload()
CLEAN = next(Path('.').glob('cleaned_train*.zip'))
RAW = Path('Student folder.zip')
clean_zip = zipfile.ZipFile(CLEAN)
raw_zip = zipfile.ZipFile(RAW)
def trip_files(z, partition, suffix):
    return {Path(n).stem:n for n in z.namelist() if n.endswith(suffix) and partition in Path(n).parts}
clean_tracks = trip_files(clean_zip, 'cleaned_train', '.track')
train_tracks = trip_files(raw_zip, 'train', '.track')
test_tracks = trip_files(raw_zip, 'student_test', '.track')
assert set(clean_tracks) == set(train_tracks), 'Cleaned and original training IDs differ'
assert not set(train_tracks) & set(test_tracks)
print('Training trips:', len(clean_tracks), '| Student test trips:',len(test_tracks))
def read_array(z, name, dtype=float):
    return np.loadtxt(io.BytesIO(z.read(name)), dtype=dtype, ndmin=2)
def load_trip(tid, partition):
    tracks = train_tracks if partition=='train' else test_tracks
    base = tracks[tid].rsplit('.',1)[0]
    track = read_array(clean_zip, clean_tracks[tid]) if partition=='train' else read_array(raw_zip,tracks[tid])
    nodes = read_array(raw_zip,base+'.nodes')
    arcs = read_array(raw_zip,base+'.arcs',int)
    route = read_array(raw_zip,base+'.route',int).ravel()
    assert track.shape[1]==3 and nodes.shape[1]==2 and arcs.shape[1]==2
    assert arcs.min()>=0 and arcs.max()<len(nodes) and route.min()>=0 and route.max()<len(arcs)
    track=track[np.isfinite(track).all(axis=1)]
    track=track[np.argsort(track[:,2],kind='stable')]
    track=track[np.r_[True,np.diff(track[:,2])>0]]
    return track,nodes,arcs,route

## Candidate generation and feature engineering
Convert longitude/latitude to local metres using an equirectangular approximation centred on each trip. Index regularly sampled directed road segments using a KD-tree; retrieve nearby segments and rerank them by exact point-to-segment distance. Features use geometry, direction, speed, local GPS context and candidate rank. No reference route, trip ID or arc ID is a feature.

In [ ]:
def project(lonlat, origin):
    scale=np.array([111195*np.cos(np.deg2rad(origin[1])),111195])
    return (lonlat-origin)*scale

def geometry(p,a,b):
    v=b-a
    length=np.linalg.norm(v,axis=-1)
    u=np.clip(np.sum((p-a)*v,axis=-1)/np.maximum(length**2,1e-9),0,1)
    q=a+u[...,None]*v
    return np.linalg.norm(p-q,axis=-1),u,length,q

def index_segments(a,b):
    count=np.maximum(2,np.ceil(np.linalg.norm(b-a,axis=1)/20).astype(int)+1)
    ids=np.repeat(np.arange(len(a)),count)
    offsets=np.cumsum(count)-count
    frac=(np.arange(len(ids))-np.repeat(offsets,count))/np.repeat(count-1,count)
    samples=a[ids]+frac[:,None]*(b[ids]-a[ids])
    return cKDTree(samples),ids

def candidates(points,a,b,k):
    k=min(k,len(a)); tree,ids=index_segments(a,b)
    query_n=min(96,len(ids))
    result=np.empty((len(points),k),dtype=np.int32)
    for start in range(0,len(points),1024):
        chunk=points[start:start+1024]
        _,idx=tree.query(chunk,k=query_n,workers=N_JOBS)
        idx=np.asarray(idx).reshape(len(chunk),query_n)
        for j,p in enumerate(chunk):
            pool=np.unique(ids[idx[j]])
            if len(pool)<k: pool=np.arange(len(a))
            d,*_=geometry(p,a[pool],b[pool])
            order=np.argsort(d,kind='stable')[:k]
            result[start+j]=pool[order]
    return result

FEATURES=['distance_m','distance_rank','distance_gap_m','segment_length_m','projection_fraction','heading_cos','speed_mps','time_gap_s','prev_distance_m','next_distance_m','prev_heading_cos','next_heading_cos','endpoint_distance_m','step_m','local_turn_cos']

def unit(v):
    return v/np.maximum(np.linalg.norm(v,axis=-1,keepdims=True),1e-9)

def build_features(track,nodes,arcs,stride=1):
    origin=np.median(track[:,:2],axis=0)
    points=project(track[:,:2],origin); xy=project(nodes,origin)
    a,b=xy[arcs[:,0]],xy[arcs[:,1]]
    prev=np.vstack([points[0],points[:-1]]); nxt=np.vstack([points[1:],points[-1]])
    vin=points-prev; vout=nxt-points
    step=np.linalg.norm(vin,axis=1); dt=np.r_[1,np.diff(track[:,2])]
    speed=step/np.maximum(dt,1e-3)
    take=np.arange(0,len(points),stride)
    p=points[take]; cand=candidates(p,a,b,K)
    aa,bb=a[cand],b[cand]; d,u,length,q=geometry(p[:,None,:],aa,bb)
    direction=unit(bb-aa)
    def cos(v): return np.sum(unit(v[take])[:,None,:]*direction,axis=-1)
    prev_d=geometry(prev[take,None,:],aa,bb)[0]
    next_d=geometry(nxt[take,None,:],aa,bb)[0]
    def broad(v): return np.broadcast_to(v[take,None],d.shape)
    matrix=np.stack([d,np.broadcast_to(np.arange(cand.shape[1]),d.shape),d-d[:,:1],length,u,cos(vin+vout),broad(speed),broad(dt),prev_d,next_d,cos(vin),cos(vout),np.minimum(u,1-u)*length,broad(step),broad(np.sum(unit(vin)*unit(vout),axis=-1))],axis=-1).astype(np.float32)
    return matrix,cand,take,points,a,b

def reference_labels(points,a,b,route,take):
    # Restrict label assignment to supplied reference arcs; independent of candidate generation.
    # Nearest reference arc with a modest direction penalty; stationary points use distance alone.
    ids=np.unique(route)
    prev=np.vstack([points[0],points[:-1]])
    nxt=np.vstack([points[1:],points[-1]])
    p=points[take]; nearby=candidates(p,a[ids],b[ids],min(12,len(ids)))
    r=ids[nearby]; aa,bb=a[r],b[r]
    d=geometry(p[:,None,:],aa,bb)[0]
    v=(nxt-prev)[take]; heading=np.sum(unit(v)[:,None,:]*unit(bb-aa),axis=-1)
    penalty=np.where(np.linalg.norm(v,axis=1)[:,None]>3,3*(1-heading),0)
    return r[np.arange(len(p)),np.argmin(d+penalty,axis=1)]

In [ ]:
# Smoke test the actual data and inspect candidate coverage before training.
tid=next(iter(clean_tracks)); track,nodes,arcs,route=load_trip(tid,'train')
F,C,I,P,A,B=build_features(track,nodes,arcs,10)
Y=reference_labels(P,A,B,route,I)
print(tid, F.shape, 'candidate recall:',np.mean((C==Y[:,None]).any(axis=1)))
assert np.isfinite(F).all()
plt.figure(figsize=(7,5)); plt.plot(track[:,0],track[:,1],lw=1); plt.xlabel('Longitude'); plt.ylabel('Latitude'); plt.title('Cleaned training GPS example'); plt.show()

In [ ]:
def prepare(partition,stride):
    output=[]
    ids=sorted(clean_tracks if partition=='train' else test_tracks)
    for i,tid in enumerate(ids):
        track,nodes,arcs,route=load_trip(tid,partition)
        F,C,I,P,A,B=build_features(track,nodes,arcs,stride)
        Y=reference_labels(P,A,B,route,I)
        output.append(dict(tid=tid,X=F,candidates=C,ref=Y,arcs=arcs))
        if (i+1)%10==0: print(partition,i+1,'/',len(ids))
    return output

train=prepare('train',TRAIN_STRIDE)
student=prepare('student_test',1)
def flatten(records):
    X=np.concatenate([r['X'].reshape(-1,len(FEATURES)) for r in records])
    y=np.concatenate([(r['candidates']==r['ref'][:,None]).reshape(-1) for r in records]).astype(np.int32)
    return X,y
def coverage(records):
    return np.mean(np.concatenate([(r['candidates']==r['ref'][:,None]).any(axis=1) for r in records]))
print('Train observations:',sum(len(r['ref']) for r in train),'Candidate recall:',coverage(train))
print('Student observations:',sum(len(r['ref']) for r in student),'Candidate recall:',coverage(student))

## Train and evaluate XGBoost
Candidate-level binary macro F1 measures both match and non-match classes. Observation-level top-1 accuracy measures whether the highest-scoring candidate equals the proxy reference arc; candidate misses count as errors. Route continuity here means consecutive predictions are identical or directly connected, a strict diagnostic that may penalize legitimate skipped arcs. Group-aware CV keeps entire trips together. The class weight is calculated from training-fold labels only.

In [ ]:
BASE = dict(n_estimators=180,max_depth=5,learning_rate=0.1,subsample=0.85,colsample_bytree=0.85,min_child_weight=3,reg_lambda=2)
def fit(records,params):
    X,y=flatten(records)
    weight=float((y==0).sum()/max((y==1).sum(),1))
    model=XGBClassifier(**params,objective='binary:logistic',eval_metric='logloss',tree_method='hist',random_state=SEED,n_jobs=N_JOBS,scale_pos_weight=weight)
    model.fit(X,y)
    return model

def evaluate(model,records,details=False):
    start=time.perf_counter(); X,y=flatten(records)
    p=model.predict_proba(X)[:,1]; elapsed=time.perf_counter()-start
    top=[]; cont=[]; cursor=0
    for r in records:
        c=r['candidates']; size=c.size; prob=p[cursor:cursor+size].reshape(c.shape); cursor+=size
        pred=c[np.arange(len(c)),prob.argmax(axis=1)]
        top.extend(pred==r['ref'])
        if len(pred)>1:
            cont.extend((pred[:-1]==pred[1:]) | (r['arcs'][pred[:-1],1]==r['arcs'][pred[1:],0]))
    hat=(p>=0.5).astype(int)
    metrics=dict(candidate_accuracy=accuracy_score(y,hat),macro_f1=f1_score(y,hat,average='macro',zero_division=0),precision=precision_score(y,hat,zero_division=0),recall=recall_score(y,hat,zero_division=0),log_loss=log_loss(y,p,labels=[0,1]),top1_accuracy=float(np.mean(top)),candidate_miss_rate=float(1-coverage(records)),route_continuity=float(np.mean(cont)) if cont else np.nan,inference_seconds=elapsed)
    return (metrics,y,hat) if details else metrics

splits=list(GroupKFold(n_splits=3).split(train,groups=[r['tid'] for r in train]))
def cv(params):
    rows=[]
    for tr,va in splits:
        m=fit([train[i] for i in tr],params)
        rows.append(evaluate(m,[train[i] for i in va]))
    return pd.DataFrame(rows).mean().to_dict()

start=time.perf_counter(); baseline_cv=cv(BASE); baseline=fit(train,BASE)
baseline_time=time.perf_counter()-start
print('Baseline CV:',baseline_cv)

In [ ]:
# Bayesian sequential model-based optimization (Optuna TPE).
# Optimize group-CV macro F1; keep student-test untouched during search.
def objective(trial):
    params=dict(n_estimators=trial.suggest_int('n_estimators',120,300,step=60),max_depth=trial.suggest_int('max_depth',4,8),learning_rate=trial.suggest_float('learning_rate',0.035,0.18,log=True),subsample=trial.suggest_float('subsample',0.7,1.0),colsample_bytree=trial.suggest_float('colsample_bytree',0.65,1.0),min_child_weight=trial.suggest_int('min_child_weight',1,12),reg_lambda=trial.suggest_float('reg_lambda',0.5,12,log=True))
    scores=cv(params)
    trial.set_user_attr('cv_top1_accuracy',scores['top1_accuracy'])
    return scores['macro_f1']
study=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=SEED,n_startup_trials=3))
study.enqueue_trial(BASE)
start=time.perf_counter(); study.optimize(objective,n_trials=N_TRIALS)
tuning_seconds=time.perf_counter()-start
study.trials_dataframe().to_csv(OUT/'bayesian_trials.csv',index=False)
print('Best parameters:',study.best_params,'CV macro F1:',study.best_value)
start=time.perf_counter(); tuned=fit(train,study.best_params); tuned_time=time.perf_counter()-start

In [ ]:
# Student-test evaluation after tuning; no test-driven parameter updates.
base_metrics=evaluate(baseline,student)
tuned_metrics,y_true,y_pred=evaluate(tuned,student,details=True)
comparison=pd.DataFrame([dict(model='XGBoost baseline',cv_macro_f1=baseline_cv['macro_f1'],training_and_cv_seconds=baseline_time,**base_metrics),dict(model='XGBoost tuned',cv_macro_f1=study.best_value,training_and_cv_seconds=tuned_time,**tuned_metrics)])
display(comparison.T)
comparison.to_csv(OUT/'model_comparison.csv',index=False)
comparison.set_index('model')[['macro_f1','top1_accuracy','cv_macro_f1']].plot.bar(rot=0,ylim=(0,1),title='Measured model comparison'); plt.tight_layout(); plt.savefig(OUT/'comparison.png'); plt.show()
ConfusionMatrixDisplay(confusion_matrix(y_true,y_pred,labels=[0,1]),display_labels=['Non-match','Match']).plot(); plt.title('Tuned XGBoost: candidate-level confusion matrix'); plt.savefig(OUT/'confusion_matrix.png'); plt.show()
pd.Series(tuned.feature_importances_,index=FEATURES).sort_values().plot.barh(figsize=(8,6),title='XGBoost feature importance'); plt.tight_layout(); plt.savefig(OUT/'feature_importance.png'); plt.show()

In [ ]:
# Export the model selected using training CV, not student-test scores.
use_tuned=study.best_value>baseline_cv['macro_f1']
model=tuned if use_tuned else baseline
model.save_model(str(OUT/'xgboost_model.json'))
config=dict(features=FEATURES,K=K,train_stride=TRAIN_STRIDE,seed=SEED,params=study.best_params if use_tuned else BASE,selected='tuned' if use_tuned else 'baseline',tuning_seconds=tuning_seconds,label_method='nearest reference route arc with heading tie-break',candidate_sampling_m=20)
(OUT/'config.json').write_text(json.dumps(config,indent=2))
print('Saved:',config['selected'])
def predict_trip(track,nodes,arcs):
    # No route argument: inference uses GPS + network only.
    X,C,I,*_=build_features(track,nodes,arcs,1)
    p=model.predict_proba(X.reshape(-1,len(FEATURES)))[:,1].reshape(C.shape)
    return C[np.arange(len(C)),p.argmax(axis=1)],C,p

import shutil
archive=shutil.make_archive('pathsnap_xgb_results','zip',OUT)
files.download(archive)

## Next steps for the hackathon
Inspect candidate miss rate first: even a perfect classifier cannot recover a reference arc excluded from its candidates. Compare the same metrics for at least four other model families before final selection. Feed the selected model's candidate scores into HMM/Viterbi, and compare classifier-only and decoded predictions on permitted student-test data. Retain the notebook, experiment logs, model JSON, config and inference implementation for the GitHub submission. Do not claim a score until your run produces it.